In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('/files/Projet_Barca/')


from Team_Data_Loader import clear_key_player_data 
from Team_Data_Loader import validate_players_data
from Team_Data_Loader import load_raw_key_players_data


    
def create_advanced_measures(players_df):
    # Creating some other features for the players
    
    # CORR: To be sure that everything is correct we convert comas into points
    numeric_cols_to_convert = ['xG', 'PrgC', 'PrgP', 'Pass', 'Shots', 'Take_On', 'Tkl', 'TklW', 'Int', 'Recov', 
                               'Min', 'MP', 'Starts', 'Succes_P%', 'Sot%', 'TO%']
    
    for col in numeric_cols_to_convert:
        if col in players_df.columns and players_df[col].dtype == 'object':
            players_df[col] = players_df[col].astype(str).str.replace(',', '.').astype(float)
            
    players_df['Stamina']=(players_df['Min']/ players_df['MP'])
    
    # Per 90 minutes statistics (essential for comparison specially for football because a game has a duration of 90 mins)
    players_df['Goals_p90'] = (players_df['Goals'] / players_df['Min']) * 90
    players_df['Assists_p90'] = (players_df['Assists'] / players_df['Min']) * 90
    players_df['xG_p90'] = (players_df['xG'] / players_df['Min']) * 90
    players_df['Shots_p90'] = (players_df['Shots'] / players_df['Min']) * 90
    players_df['Shots_on_target_p90'] = (players_df['Shots'] * players_df['Sot%']  / players_df['Min']) * 90 /100
    
    # Offensive Efficiency Measures
    players_df['Conversion_Rate'] = (players_df['Goals'] / players_df['Shots'] * 100).round(1)
    players_df['Goal_Contribution_p90'] = players_df['Goals_p90'] + players_df['Assists_p90']
    players_df['xG_efficiency'] = (players_df['Goals'] / players_df['xG']).round(2)
    
    # Creative impact Measures
    players_df['Progressive_Passes_p90'] = (players_df['PrgP'] / players_df['Min']) * 90
    players_df['Progressive_Carries_p90'] = (players_df['PrgC'] / players_df['Min']) * 90
    players_df['Total_Progressive_Actions_p90'] = players_df['Progressive_Passes_p90'] + players_df['Progressive_Carries_p90']
    
    # Involvement measures
    players_df['Passes_p90'] = (players_df['Pass'] / players_df['Min']) * 90
    players_df['Successful_Passes_p90'] = (players_df['Pass'] * players_df['Succes_P%']  / players_df['Min']) * 90 / 100
    
    # Defensive measures per 90 min
    players_df['Tackles_p90'] = (players_df['Tkl'] / players_df['Min']) * 90
    players_df['Successful_Tackles_p90'] = (players_df['TklW'] / players_df['Min']) * 90
    players_df['Interceptions_p90'] = (players_df['Int'] / players_df['Min']) * 90
    players_df['Ball_Recoveries_p90'] = (players_df['Recov'] / players_df['Min']) * 90
    
    # Dribbling and 1v1 measures
    players_df['Take_Ons_p90'] = (players_df['Take_On'] / players_df['Min']) * 90
    players_df['Successful_Take_Ons_p90'] = (players_df['Take_On'] * players_df['TO%']  / players_df['Min']) * 90 /100
    
    # Some measures that are interistics for clustering
    players_df['Versality_index'] = (players_df['Goal_Contribution_p90'] + players_df['Total_Progressive_Actions_p90']+players_df['Tackles_p90']).round(2)
    
    players_df['Attack_Defense_Ratio'] = np.where(players_df['Tackles_p90'] > 0, players_df['Goal_Contribution_p90'] / players_df['Tackles_p90'], 0).round(2)
                                                  
    # Player role classification
    players_df['Player_Role'] = players_df.apply(classify_player_role, axis=1)
    
    # Overall impact score (composite metric)
    players_df['Overall_Impact_Score'] = players_df.apply(calculate_impact_score, axis=1)
    
    """ I did add this safety because of the function: final_data_formatting for getting rid of the errors and convertir number into float"""
    
    new_metrics = ['Goals_p90', 'Assists_p90', 'xG_p90', 'Shots_p90', 'Shots_on_target_p90',
                   'Conversion_Rate', 'Goal_Contribution_p90', 'xG_efficiency',
                   'Progressive_Passes_p90', 'Progressive_Carries_p90', 'Total_Progressive_Actions_p90',
                   'Passes_p90', 'Successful_Passes_p90', 'Tackles_p90', 'Successful_Tackles_p90',
                   'Interceptions_p90', 'Ball_Recoveries_p90', 'Take_Ons_p90', 'Successful_Take_Ons_p90',
                   'Versatility_Index', 'Attack_Defense_Ratio', 'Overall_Impact_Score']
    for metric in new_metrics:
        if metric in players_df.columns:
            players_df[metric] = pd.to_numeric(players_df[metric], errors='coerce').fillna(0)
    
    print("Advanced metrics creation completed")
    
    return players_df

def classify_player_role(row):
    #Classifying the different players role into the team
    
    if row['Pos'] == 'DF':
        return 'Defender'
    elif row['Pos'] == 'MF':
        if row['Assists_p90'] > row['Goals_p90']:
            return 'Creative Midfielder'
        else:
            return 'Box-to-Box Midfielder'
    elif row['Pos'] == 'FW':
        if row['Goals_p90'] > 0.4:
            return 'Goalscorer'
        elif row['Assists_p90'] > 0.3:
            return 'Playmaker'
        else:
            return 'Winger'
    return 'Utility Player'

def calculate_impact_score(row):
    """Calculate impact score for a single player based on their role and like a Fifa card with a score """
    
    metrics_to_include = [
        'Goals_p90', 'Successful_Take_Ons_p90', 'Progressive_Passes_p90', 
         'Successful_Tackles_p90', 'Stamina']
    
    scores = []
    for metric in metrics_to_include:
        if metric in row.index:
            value = row[metric]
            if pd.isna(value) or value is None:
                scores.append(0.0)
            else:
                scores.append(float(value))   # useful for avoiding some errors due to values types
            
    
    if scores and len(scores) == len(metrics_to_include):
        """"Role-specific weights like in fifa because you can't judge a defender and a forward on the same basis of criteria and I
        just kept the roles of goalscorer, playmaker, creative midfielder and defender because they are the output of the precedent
        function that I seen in my precedent files"""
        
        player_role = row['Player_Role']
        
        if player_role == 'Goalscorer':
            weights = [0.4, 0.2, 0.2, 0.1, 0.1]
        elif player_role == 'Playmaker':
            weights = [0.3, 0.4, 0.3, 0.1, 0.1]
        elif player_role == 'Creative Midfielder':
            weights = [0.05, 0.3, 0.3, 0.2, 0.15]
        elif player_role == 'Defender':
            weights = [0.05, 0.15, 0.3, 0.3, 0.2]
        else:
            weights = [0.25, 0.25, 0.2, 0.2, 0.1]
            
        weighted_score = sum(score * weight for score, weight in zip(scores, weights))
        
        result = round(weighted_score , 1)
        
    
        return result






def final_data_formatting(players_df):
    #Final formatting and cleaning
    
    # Round numeric columns for readability
    rounding_rules = {
        'Goals_p90': 2,
        'Assists_p90': 2,
        'xG_p90': 2,
        'Shots_p90': 1,
        'Conversion_Rate': 1,
        'Goal_Contribution_p90': 2,
        'Progressive_Passes_p90': 1,
        'Progressive_Carries_p90': 1,
        'Total_Progressive_Actions_p90': 1,
        'Tackles_p90': 2,
        'Interceptions_p90': 2,
        'Overall_Impact_Score': 1
    }
    
    for col, decimals in rounding_rules.items():
        if col in players_df.columns:
            players_df[col] = players_df[col].round(decimals)
    
    # Handle in case of infinite values from division
    players_df['xG_efficiency'] = players_df['xG_efficiency'].replace([np.inf, -np.inf], 0)
    
    # Sort by position and impact score
    players_df = players_df.sort_values(['Pos', 'Overall_Impact_Score'], ascending=[True, False])
    
    # Reset index for clean output
    players_df = players_df.reset_index(drop=True)
    
    return players_df


def get_player_statistics_summary(players_df):
    """Print  summary without returning data because it wasn't beautiful with the returning of data"""
    
    df_display = players_df.copy()
    
    print("\n" + "="*70)
    print("Player analysis report")
    print("="*70)
    
    print(f"\nRoster: {len(df_display)} players analyzed ")
    print(f"Positions: {df_display['Pos'].value_counts().to_dict()}")
    print(f"Roles: {df_display['Player_Role'].value_counts().to_dict()}")
    
    print("\nAvg performances by positions:")
    position_stats = df_display.groupby('Pos').agg({
        'Goals_p90': 'mean', 'Assists_p90': 'mean', 'Overall_Impact_Score': 'mean'
    }).round(2)
    print(position_stats.to_string())
    
    print("\nTop performers:")
    print(f"   - Most Goals: {df_display.loc[df_display['Goals_p90'].idxmax()]['Players']}")
    print(f"   - Most Assists: {df_display.loc[df_display['Assists_p90'].idxmax()]['Players']}")
    print(f"   - Most Creative: {df_display.loc[df_display['Progressive_Passes_p90'].idxmax()]['Players']}")
    print(f"   - Best Defender: {df_display.loc[df_display['Successful_Tackles_p90'].idxmax()]['Players']}")
    print(f"   - Highest Impact: {df_display.loc[df_display['Overall_Impact_Score'].idxmax()]['Players']}")
    
    print("\nGlobal efficiency:")
    print(f"   - Avg Conversion Rate: {df_display['Conversion_Rate'].mean().round(1)}")
    print(f"   - Avg Passing Accuracy: {df_display['Succes_P%'].mean().round(1)}")
    print(f"   - Avg Goal Contribution: {df_display['Goal_Contribution_p90'].mean().round(2)}")
    
    print("\n" + "="*70)
    
    return summary

def display_metrics_table(players_df):
    """Affiche un tableau simple avec les métriques importantes"""
    
    # Sélectionner les métriques à afficher
    metrics_to_display = [
        'Players', 'Pos', 'Player_Role', 
        'Goals_p90', 'Assists_p90', 'Goal_Contribution_p90',
        'Progressive_Passes_p90', 'Successful_Tackles_p90',
        'Successful_Take_Ons_p90', 'Stamina', 'Overall_Impact_Score'
    ]
    
    # Garder seulement les colonnes existantes
    available_metrics = [m for m in metrics_to_display if m in players_df.columns]
    
    # Créer le tableau
    table_df = players_df[available_metrics].copy()
    
    print("TABLEAU DES MÉTRIQUES DES JOUEURS")
    print("=" * 100)
    print(table_df.to_string(index=False))
    print("=" * 100)

if __name__ == "__main__":
    # Charge les données
    players_raw_df = load_raw_key_players_data()
    
    
    players_analyzed = create_advanced_measures(players_raw_df)
    
    
    players_final = final_data_formatting(players_analyzed)
    
    display_metrics_table(players_final)
    
    
    summary = get_player_statistics_summary(players_final)
    




Key Players Data analysis
Raw dataset shape: 4 players, 20 metrics
Available columns: ['Players', 'Pos', 'MP', 'Starts', 'Min', 'Goals', 'Assists', 'xG', 'PrgC', 'PrgP', 'Pass', 'Succes_P%', 'Shots', 'Sot%', 'Take_On', 'TO%', 'Tkl', 'TklW', 'Int', 'Recov']

First rows of cleaned data:
    Players Pos  MP  Starts   Min  Goals  Assists    xG  PrgC  PrgP  Pass  \
0     Pedri  MF  37      35  2879      4        5   2,2    92   360  2749   
1   Raphina  FW  36      32  2839     18        9  19,2    94   135  1497   
2     Yamal  FW  35      31  2856      9       13   9,8   181   160  1553   
3  Martinez  DF  28      28  2490      0        4   0,5    70   260  2435   

  Succes_P%  Shots  Sot%  Take_On   TO%  Tkl  TklW  Int  Recov  
0      87,4     25    44       61  65,6   61    34   26    254  
1      71,1    112  35,7      103  50,5   36    22   11     94  
2      74,8    144  31,9      316  50,9   42    28   16    123  
3        91     13  15,4        3   100   25    14   18    126  

-